## 1. Import Required Libraries

Import the necessary libraries for inference, including basic libraries, deep learning frameworks, and model-related dependencies.

In [2]:
import os
import sys
import cv2
import random
import warnings
from typing import Dict, List, Optional, Tuple, Union
from datetime import datetime

# Deep learning related libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm import tqdm

# Visualization and analysis libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ignore warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducible results
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("✅ All dependency libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

✅ All dependency libraries imported successfully!
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
GPU name: NVIDIA GeForce RTX 3060


## 1. Import Required Libraries

Import the necessary libraries for inference, including basic libraries, deep learning frameworks, and model-related dependencies.

In [3]:
import os
import sys
import cv2
import random
import warnings
from typing import Dict, List, Optional, Tuple, Union
from datetime import datetime

# Deep learning related libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm import tqdm

# Visualization and analysis libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ignore warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducible results
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("✅ All dependency libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

✅ All dependency libraries imported successfully!
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
GPU name: NVIDIA GeForce RTX 3060


In [4]:
# =============================================================================
# Configuration Parameter Settings
# =============================================================================

# Default inference configuration
DEFAULT_CONFIG = {
    'NAME': 'Amair',                        # Identity ID
    'DEVICE': '0',                  # CUDA device ID
    'OPTIM_FACECHECKER': False,     # Whether to optimize face checker
    'TOKEN_NUM': 32,                # Number of VIP tokens
    'TOTAL_NUM': 50,                # Number of images tested per method (reduced for quick testing)
    'MAX_NEW_TOKENS': 4096,         # Maximum generated tokens
    'FACE_IMG_SIZE': (112, 112),    # Face image size
    'FACE_TOKEN_DIM': 3584,         # Face token dimension
    'PRETRAIN_PATH': './checkpoints/checkpoints_attr_Stage2_merge',  # Pretrained model path (consistent with training stage)
    'MAX_IMAGE_SIZE': 448,          # Maximum image size (consistent with training stage)
    'DEFAULT_LR': 1.0,              # Default learning rate (consistent with training stage)
    'WEIGHT_DECAY': 1e-3,           # Weight decay (consistent with training stage)
}


# Inference modes
INFERENCE_MODES = ['cls', 'exp', 'reasoning']

# Set CUDA environment
os.environ['CUDA_VISIBLE_DEVICES'] = DEFAULT_CONFIG['DEVICE']

print("📝 Configuration parameter setup complete!")
print(f"Current configuration: {DEFAULT_CONFIG}")
print(f"Inference modes: {INFERENCE_MODES}")

# Set CUDA environment
os.environ['CUDA_VISIBLE_DEVICES'] = DEFAULT_CONFIG['DEVICE']

📝 Configuration parameter setup complete!
Current configuration: {'NAME': 'Amair', 'DEVICE': '0', 'OPTIM_FACECHECKER': False, 'TOKEN_NUM': 32, 'TOTAL_NUM': 50, 'MAX_NEW_TOKENS': 4096, 'FACE_IMG_SIZE': (112, 112), 'FACE_TOKEN_DIM': 3584, 'PRETRAIN_PATH': './checkpoints/checkpoints_attr_Stage2_merge', 'MAX_IMAGE_SIZE': 448, 'DEFAULT_LR': 1.0, 'WEIGHT_DECAY': 0.001}
Inference modes: ['cls', 'exp', 'reasoning']


## 2. Auxiliary Function Definitions

Define auxiliary functions for image processing, similarity calculation, and message generation.

In [5]:
def get_message(image: str, sim_score: int, mode: str = 'cls') -> List[Dict]:
    """
    Generate message templates for different inference modes
    
    Args:
        image (str): Image file path
        sim_score (int): Face similarity score (0-100)
        mode (str): Inference mode - 'cls', 'reasoning', or 'exp'
        
    Returns:
        List[Dict]: Model message templates
    """
    # Base message template containing face padding tokens and images
    base_content = [
        {"type": "text", "text": "<|face_pad|>"},
        {"type": "image", "image": image}
    ]

    # Select prompt text based on mode
    if mode == 'reasoning':
        prompt_text = (
            "Please determine whether the person in the input image is VIP user. The face similarity of these two face is {}/100. The face tokens are shown as follows, <|face_pad|> . You need to make a step-by-step judgment based on different facial attributes and provide your conclusion.".format(sim_score)
        )
    elif mode == 'exp':
        prompt_text = (
            "Please determine whether the person in the input image is VIP user. The face similarity of these two face is {}/100. The face tokens are shown as follows, <|face_pad|> . You should first give your answer by 'yes' or 'no'. Then, you should explain your reasoning step by step based on different facial attributes.".format(sim_score)
        )
    elif mode == 'cls':
        prompt_text = (
            "Please determine whether the person in the input image is VIP user. The face similarity of these two face is {}/100. The face tokens are shown as follows, <|face_pad|> . You should directly answer 'yes' or 'no' without any explanation.".format(sim_score)
        )
    else:
        raise ValueError(f"Unsupported mode: {mode}")
    
    base_content.append({"type": "text", "text": prompt_text})
    
    return [{"role": "user", "content": base_content}]


@torch.no_grad()
def preprocess_face_image(img_path: str) -> torch.Tensor:
    """
    Preprocess face image for feature extraction
    
    Args:
        img_path (str): Input image path
        
    Returns:
        torch.Tensor: Preprocessed image tensor
    """
    # Load and resize image
    img = cv2.imread(img_path)
    img = cv2.resize(img, DEFAULT_CONFIG['FACE_IMG_SIZE'])
    
    # Convert BGR to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Convert to CHW format
    img = img.transpose(2, 0, 1)
    
    # Convert to tensor and normalize
    img = torch.tensor(img).unsqueeze(0).float()
    img.div_(255).sub_(0.5).div_(0.5)  # Normalize to [-1, 1]
    
    return img


@torch.no_grad()
def extract_face_embedding(img_path: str, face_model: torch.nn.Module) -> torch.Tensor:
    """
    Extract face embedding from image
    
    Args:
        img_path (str): Input image path
        face_model (torch.nn.Module): Face recognition model
        
    Returns:
        torch.Tensor: Face embedding vector
    """
    face_model.eval()
    
    # Get device ID
    device_id = int(DEFAULT_CONFIG['DEVICE'])
    device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
    
    # Preprocess image
    img = preprocess_face_image(img_path)
    img = img.to(device)  # Move image to correct device
    
    # Extract embedding
    emb = face_model.face_emb_forward(img)
    emb = emb.squeeze().detach().cpu()
    
    return emb


def calculate_similarity_with_center(img_path: str, emb_center: torch.Tensor, face_model: torch.nn.Module) -> int:
    """
    Calculate similarity between image and center embedding
    
    Args:
        img_path (str): Input image path
        emb_center (torch.Tensor): Center embedding tensor
        face_model (torch.nn.Module): Face recognition model
        
    Returns:
        int: Similarity score (0-100)
    """
    # Get device ID
    device_id = int(DEFAULT_CONFIG['DEVICE'])
    device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
    
    # Extract embedding
    emb = extract_face_embedding(img_path, face_model)
    emb = F.normalize(emb, p=2, dim=0)
    
    # Calculate cosine similarity and scale to [0, 1]
    emb = emb.to(device)
    emb_center = emb_center.to(device)
    sim = 0.5 + (0.5 * torch.cosine_similarity(emb, emb_center, dim=0).item())
    
    return int(sim * 100)


print("🔧 Auxiliary functions defined successfully!")
print("Including features:")
print("- Message generation function")
print("- Image preprocessing function")
print("- Face embedding extraction function")
print("- Similarity calculation function")

🔧 Auxiliary functions defined successfully!
Including features:
- Message generation function
- Image preprocessing function
- Face embedding extraction function
- Similarity calculation function


## 3. Model Loading Function Implementation

Implement functions for loading vision-language models, processors, and Wrapper implementations.

In [6]:
def load_vip_models(config: Dict) -> Tuple[torch.nn.Module, torch.nn.Module, object]:
    """
    Load VIP DFD model-related components
    
    Args:
        config (Dict): Configuration parameters
        
    Returns:
        Tuple: (model, processor, face_model)
    """
    print("🔄 Starting to load VIP DFD models...")
    
    try:
        # Import necessary modules
        from VIP_Dataset import VIP_Dataset
        from Models.VIPGuard import Qwen2_5_VLForConditionalGeneration, Qwen2_5_VLProcessor
        from Models.Wrapper import Wrapper
        from Models.Face_Model.FaceModel import FG_Face
        
        # Pretrained model path
        pretrain_path = config['PRETRAIN_PATH']
        
        # Get specified device
        device = f"cuda:{config['DEVICE']}" if torch.cuda.is_available() else "cpu"
        print(f"📌 Using device: {device}")
        
        print("📥 Loading vision-language model...")
        # Use specified device instead of automatic assignment
        vl_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(pretrain_path, device_map={"": int(config['DEVICE'])})
        processor = Qwen2_5_VLProcessor.from_pretrained(pretrain_path)
        
        print("🔧 Initializing Wrapper model...")
        model = Wrapper(
            vl_model=vl_model,
            processor=processor,
            optim_facechecker=config['OPTIM_FACECHECKER'],
            vip_token_num=config['TOKEN_NUM']
        )
        model.cuda(int(config['DEVICE']))
        
        print("📥 Loading face recognition model...")
        face_model = FG_Face(
            attributes='',
            token_dim=config['FACE_TOKEN_DIM'],
            model_name='transface'
        )
        face_model.cuda(int(config['DEVICE']))
        
        print("✅ All models loaded successfully!")
        return model, processor, face_model
        
    except ImportError as e:
        print(f"❌ Import error: {e}")
        print("Please ensure all necessary modules are in the correct path.")
        raise
    except Exception as e:
        print(f"❌ Model loading error: {e}")
        raise


def load_vip_checkpoint(model: torch.nn.Module, config: Dict) -> torch.nn.Module:
    """
    Load VIP checkpoint
    
    Args:
        model (torch.nn.Module): VIP model
        config (Dict): Configuration parameters
        
    Returns:
        torch.nn.Module: Model after loading checkpoint
    """
    checkpoint_path = './checkpoints/Stage3/id0_train/vip_token.pt'
    
    print(f"📥 Loading VIP checkpoint: {checkpoint_path}")
    
    if not os.path.exists(checkpoint_path):
        print(f"⚠️  Warning: Checkpoint file does not exist: {checkpoint_path}")
        return model
    
    try:
        model.load_vip(checkpoint_path)
        print("✅ VIP checkpoint loaded successfully!")
    except Exception as e:
        print(f"❌ Checkpoint loading error: {e}")
        raise
    
    return model


def load_face_embedding_center(config: Dict) -> torch.Tensor:
    """
    Load face embedding center
    
    Args:
        config (Dict): Configuration parameters
        
    Returns:
        torch.Tensor: Face embedding center tensor
    """
    emb_center_path = './FaceDATA/FaceEmb_Center/id0.ckpt'
    
    print(f"📥 Loading face embedding center: {emb_center_path}")
    
    if not os.path.exists(emb_center_path):
        print(f"⚠️  Warning: Embedding center file does not exist: {emb_center_path}")
        return None
    
    try:
        # Specify loading to specific device
        device = torch.device(f"cuda:{config['DEVICE']}" if torch.cuda.is_available() else "cpu")
        emb_center = torch.load(emb_center_path, map_location=device)
        print("✅ Face embedding center loaded successfully!")
        return emb_center
    except Exception as e:
        print(f"❌ Embedding center loading error: {e}")
        raise


print("🔧 Model loading function definition complete!")
print("Including features:")
print("- VIP model loading function")
print("- Checkpoint loading function")
print("- Face embedding center loading function")

🔧 Model loading function definition complete!
Including features:
- VIP model loading function
- Checkpoint loading function
- Face embedding center loading function


## 4. Load Vision-Language Models and Processors

Execute model loading to initialize core components of the VIP DFD detection system.

In [7]:
# Load all model components
print("🚀 Starting to load VIP DFD detection system...")
print("=" * 50)

# Record start time
start_time = datetime.now()

# Load VIP models
try:
    model, processor, face_model = load_vip_models(DEFAULT_CONFIG)
    
    # Load VIP checkpoint
    model = load_vip_checkpoint(model, DEFAULT_CONFIG)
    
    # Count model parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"📊 Total trainable parameters: {total_params:,}")
    
    # Display model device information
    device_id = DEFAULT_CONFIG['DEVICE']
    print(f"📱 Model loaded on device: CUDA:{device_id}")
    
    # Verify model is on specified device
    if hasattr(model, 'device'):
        print(f"📱 Model current device: {model.device}")
    elif hasattr(model.vl_model, 'device'):
        print(f"📱 VL model current device: {model.vl_model.device}")
    
    # Record loading time
    load_time = datetime.now() - start_time
    print(f"⏱️  Model loading time: {load_time.total_seconds():.2f} seconds")
    
    print("=" * 50)
    print("✅ VIP DFD detection system loading complete!")
    
    # Display model information
    print("\n📋 Model information:")
    print(f"- Model type: VIP DFD LLM")
    print(f"- Token count: {DEFAULT_CONFIG['TOKEN_NUM']}")
    print(f"- Device: CUDA:{DEFAULT_CONFIG['DEVICE']}")
    print(f"- Face model: TransFace")
    print(f"- Trainable parameters: {total_params:,}")
    
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    print("Please check if model file paths and dependencies are correct.")
    # In actual deployment, may need to exit or provide fallback solution
    print("ℹ️  Note: In this demo environment, some model files may not be accessible.")

🚀 Starting to load VIP DFD detection system...
🔄 Starting to load VIP DFD models...
📌 Using device: cuda:0
📥 Loading vision-language model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


Loading checkpoint shards:  25%|██▌       | 1/4 [00:51<02:33, 51.29s/it]


❌ Model loading error: CUDA out of memory. Tried to allocate 260.00 MiB. GPU 0 has a total capacity of 12.00 GiB of which 0 bytes is free. Of the allocated memory 18.11 GiB is allocated by PyTorch, and 335.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
❌ Model loading failed: CUDA out of memory. Tried to allocate 260.00 MiB. GPU 0 has a total capacity of 12.00 GiB of which 0 bytes is free. Of the allocated memory 18.11 GiB is allocated by PyTorch, and 335.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Please 

## 5. Load Face Recognition Model and Center Features

Load face recognition model and corresponding center feature embeddings for similarity calculation.

In [ ]:
# Load face embedding center
print("🔄 Loading face recognition related components...")
print("=" * 50)

try:
    # Get device ID
    device_id = int(DEFAULT_CONFIG['DEVICE'])
    device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
    print(f"📌 Using device: {device}")
    
    # Load face embedding center
    face_emb_center = load_face_embedding_center(DEFAULT_CONFIG)
    
    if face_emb_center is not None:
        # Ensure embedding center is on correct device
        face_emb_center = face_emb_center.to(device)
        print(f"📊 Face embedding center dimension: {face_emb_center.shape}")
        print(f"📊 Face embedding center device: {face_emb_center.device}")
        print("✅ Face recognition components loaded successfully!")
    else:
        print("⚠️  Using simulated face embedding center for demo")
        # Create simulated embedding center for demo, directly create on specified device
        face_emb_center = torch.randn(512).to(device)  # Assume embedding dimension is 512
        print(f"📊 Simulated embedding center dimension: {face_emb_center.shape}")
        print(f"📊 Simulated embedding center device: {face_emb_center.device}")
        
    print("=" * 50)
    print("✅ Face recognition system ready!")
    
    # Display face model information
    print("\n📋 Face recognition model information:")
    print(f"- Model type: TransFace")
    print(f"- Input size: {DEFAULT_CONFIG['FACE_IMG_SIZE']}")
    print(f"- Feature dimension: {DEFAULT_CONFIG['FACE_TOKEN_DIM']}")
    print(f"- Center embedding dimension: {face_emb_center.shape}")
    print(f"- Running device: {device}")
    
except Exception as e:
    print(f"❌ Face recognition component loading failed: {e}")
    print("ℹ️  Note: In this demo environment, some files may not be accessible.")

## 6. Inference Testing Function Implementation

Implement core inference testing functions for deepfake detection on images.

In [ ]:
@torch.no_grad()
def infer_single_image(img_path: str, model: torch.nn.Module, face_model: torch.nn.Module,
                      emb_center: torch.Tensor, processor, mode: str = 'cls') -> Dict:
    """
    Perform inference on a single image
    
    Args:
        img_path (str): Image path
        model (torch.nn.Module): VIP model
        face_model (torch.nn.Module): Face model
        emb_center (torch.Tensor): Center embedding
        processor: Processor
        mode (str): Inference mode
        
    Returns:
        Dict: Inference results
    """
    try:
        # Import necessary modules
        from qwen_vl_utils import process_vision_info
        
        # Get device ID
        device_id = int(DEFAULT_CONFIG['DEVICE'])
        device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
        
        # Calculate similarity score
        sim_score = calculate_similarity_with_center(img_path, emb_center, face_model)
        print(sim_score)
        # Generate messages
        messages = get_message(img_path, sim_score, mode=mode)
        
        # Prepare visual inputs
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        # If image exceeds maximum size, perform scaling (consistent with training stage)
        for i in range(len(image_inputs)):
            h, w = image_inputs[i].size
            if max(h, w) > DEFAULT_CONFIG['MAX_IMAGE_SIZE']:
                scale = DEFAULT_CONFIG['MAX_IMAGE_SIZE'] / max(h, w)
                new_w = int(w * scale)
                new_h = int(h * scale)
                image_inputs[i] = image_inputs[i].resize((new_w, new_h), Image.LANCZOS)
        
        # Prepare inference inputs
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
            our_token=True,
            our_token_length=1,
            face_pad=True,
            face_length=model.vl_model.facechecker.face_checker.vip_prompt.data.shape[0]
        )
        # Ensure inputs are on correct device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate output
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=DEFAULT_CONFIG['MAX_NEW_TOKENS'],
            do_sample=True
        )
        
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        
        # Parse results
        result_text = output_text[0]
        cls_content = result_text.split('</Conclusion>')[0]
        
        # Judge results
        prediction = None
        if 'yes' in cls_content.lower() or 'same' in cls_content.lower() or '是' in cls_content.lower():
            prediction = 'same'
        elif 'no' in cls_content.lower() or 'different' in cls_content.lower() or '否' in cls_content.lower():
            prediction = 'different'
        
        return {
            'image_path': img_path,
            'similarity_score': sim_score,
            'prediction': prediction,
            'raw_output': result_text,
            'success': True
        }
        
    except Exception as e:
        return {
            'image_path': img_path,
            'similarity_score': 0,
            'prediction': None,
            'raw_output': str(e),
            'success': False
        }

model=''
processor=''
from Models.Face_Model.FaceModel import FG_Face
face_model = FG_Face(
    attributes='',
    token_dim=DEFAULT_CONFIG['FACE_TOKEN_DIM'],
    model_name='transface'
)
face_model.cuda(int(DEFAULT_CONFIG['DEVICE']))
        
infer_single_image(img_path='./Example/with_text/id0/r_efs_i/id0_1/fake.png',
                   model=model,
                   face_model=face_model,
                   emb_center=face_emb_center,
                   processor=processor,
                   mode='cls')

In [ ]:
# =============================================================================
# Configuration Parameter Settings
# =============================================================================

# Default inference configuration
DEFAULT_CONFIG = {
    'NAME': 'Amair',                        # Identity ID
    'DEVICE': '0',                  # CUDA device ID
    'OPTIM_FACECHECKER': False,     # Whether to optimize face checker
    'TOKEN_NUM': 32,                # Number of VIP tokens
    'TOTAL_NUM': 50,                # Number of images tested per method (reduced for quick testing)
    'MAX_NEW_TOKENS': 4096,         # Maximum generated tokens
    'FACE_IMG_SIZE': (112, 112),    # Face image size
    'FACE_TOKEN_DIM': 3584,         # Face token dimension
    'PRETRAIN_PATH': './checkpoints/checkpoints_attr_Stage2_merge',  # Pretrained model path (consistent with training stage)
    'MAX_IMAGE_SIZE': 448,          # Maximum image size (consistent with training stage)
    'DEFAULT_LR': 1.0,              # Default learning rate (consistent with training stage)
    'WEIGHT_DECAY': 1e-3,           # Weight decay (consistent with training stage)
}


# Inference modes
INFERENCE_MODES = ['cls', 'exp', 'reasoning']

# Set CUDA environment
os.environ['CUDA_VISIBLE_DEVICES'] = DEFAULT_CONFIG['DEVICE']

print("📝 Configuration parameter setup complete!")
print(f"Current configuration: {DEFAULT_CONFIG}")
print(f"Inference modes: {INFERENCE_MODES}")

# Set CUDA environment
os.environ['CUDA_VISIBLE_DEVICES'] = DEFAULT_CONFIG['DEVICE']

## 2. Auxiliary Function Definitions

Define auxiliary functions for image processing, similarity calculation, and message generation.

In [ ]:
def get_message(image: str, sim_score: int, mode: str = 'cls') -> List[Dict]:
    """
    Generate message templates for different inference modes
    
    Args:
        image (str): Image file path
        sim_score (int): Face similarity score (0-100)
        mode (str): Inference mode - 'cls', 'reasoning', or 'exp'
        
    Returns:
        List[Dict]: Model message templates
    """
    # Base message template containing face padding tokens and images
    base_content = [
        {"type": "text", "text": "<|face_pad|>"},
        {"type": "image", "image": image}
    ]

    # Select prompt text based on mode
    if mode == 'reasoning':
        prompt_text = (
            "Please determine whether the person in the input image is VIP user. The face similarity of these two face is {}/100. The face tokens are shown as follows, <|face_pad|> . You need to make a step-by-step judgment based on different facial attributes and provide your conclusion.".format(sim_score)
        )
    elif mode == 'exp':
        prompt_text = (
            "Please determine whether the person in the input image is VIP user. The face similarity of these two face is {}/100. The face tokens are shown as follows, <|face_pad|> . You should first give your answer by 'yes' or 'no'. Then, you should explain your reasoning step by step based on different facial attributes.".format(sim_score)
        )
    elif mode == 'cls':
        prompt_text = (
            "Please determine whether the person in the input image is VIP user. The face similarity of these two face is {}/100. The face tokens are shown as follows, <|face_pad|> . You should directly answer 'yes' or 'no' without any explanation.".format(sim_score)
        )
    else:
        raise ValueError(f"Unsupported mode: {mode}")
    
    base_content.append({"type": "text", "text": prompt_text})
    
    return [{"role": "user", "content": base_content}]


@torch.no_grad()
def preprocess_face_image(img_path: str) -> torch.Tensor:
    """
    Preprocess face image for feature extraction
    
    Args:
        img_path (str): Input image path
        
    Returns:
        torch.Tensor: Preprocessed image tensor
    """
    # Load and resize image
    img = cv2.imread(img_path)
    img = cv2.resize(img, DEFAULT_CONFIG['FACE_IMG_SIZE'])
    
    # Convert BGR to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Convert to CHW format
    img = img.transpose(2, 0, 1)
    
    # Convert to tensor and normalize
    img = torch.tensor(img).unsqueeze(0).float()
    img.div_(255).sub_(0.5).div_(0.5)  # Normalize to [-1, 1]
    
    return img


@torch.no_grad()
def extract_face_embedding(img_path: str, face_model: torch.nn.Module) -> torch.Tensor:
    """
    Extract face embedding from image
    
    Args:
        img_path (str): Input image path
        face_model (torch.nn.Module): Face recognition model
        
    Returns:
        torch.Tensor: Face embedding vector
    """
    face_model.eval()
    
    # Get device ID
    device_id = int(DEFAULT_CONFIG['DEVICE'])
    device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
    
    # Preprocess image
    img = preprocess_face_image(img_path)
    img = img.to(device)  # Move image to correct device
    
    # Extract embedding
    emb = face_model.face_emb_forward(img)
    emb = emb.squeeze().detach().cpu()
    
    return emb


def calculate_similarity_with_center(img_path: str, emb_center: torch.Tensor, face_model: torch.nn.Module) -> int:
    """
    Calculate similarity between image and center embedding
    
    Args:
        img_path (str): Input image path
        emb_center (torch.Tensor): Center embedding tensor
        face_model (torch.nn.Module): Face recognition model
        
    Returns:
        int: Similarity score (0-100)
    """
    # Get device ID
    device_id = int(DEFAULT_CONFIG['DEVICE'])
    device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
    
    # Extract embedding
    emb = extract_face_embedding(img_path, face_model)
    emb = F.normalize(emb, p=2, dim=0)
    
    # Calculate cosine similarity and scale to [0, 1]
    emb = emb.to(device)
    emb_center = emb_center.to(device)
    sim = 0.5 + (0.5 * torch.cosine_similarity(emb, emb_center, dim=0).item())
    
    return int(sim * 100)


print("🔧 Auxiliary functions defined successfully!")
print("Including features:")
print("- Message generation function")
print("- Image preprocessing function")
print("- Face embedding extraction function")
print("- Similarity calculation function")

## 3. Model Loading Function Implementation

Implement functions for loading vision-language models, processors, and Wrapper implementations.

In [ ]:
def load_vip_models(config: Dict) -> Tuple[torch.nn.Module, torch.nn.Module, object]:
    """
    Load VIP DFD model-related components
    
    Args:
        config (Dict): Configuration parameters
        
    Returns:
        Tuple: (model, processor, face_model)
    """
    print("🔄 Starting to load VIP DFD models...")
    
    try:
        # Import necessary modules
        from VIP_Dataset import VIP_Dataset
        from Models.VIPGuard import Qwen2_5_VLForConditionalGeneration, Qwen2_5_VLProcessor
        from Models.Wrapper import Wrapper
        from Models.Face_Model.FaceModel import FG_Face
        
        # Pretrained model path
        pretrain_path = config['PRETRAIN_PATH']
        
        # Get specified device
        device = f"cuda:{config['DEVICE']}" if torch.cuda.is_available() else "cpu"
        print(f"📌 Using device: {device}")
        
        print("📥 Loading vision-language model...")
        # Use specified device instead of automatic assignment
        vl_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(pretrain_path, device_map={"": int(config['DEVICE'])})
        processor = Qwen2_5_VLProcessor.from_pretrained(pretrain_path)
        
        print("🔧 Initializing Wrapper model...")
        model = Wrapper(
            vl_model=vl_model,
            processor=processor,
            optim_facechecker=config['OPTIM_FACECHECKER'],
            vip_token_num=config['TOKEN_NUM']
        )
        model.cuda(int(config['DEVICE']))
        
        print("📥 Loading face recognition model...")
        face_model = FG_Face(
            attributes='',
            token_dim=config['FACE_TOKEN_DIM'],
            model_name='transface'
        )
        face_model.cuda(int(config['DEVICE']))
        
        print("✅ All models loaded successfully!")
        return model, processor, face_model
        
    except ImportError as e:
        print(f"❌ Import error: {e}")
        print("Please ensure all necessary modules are in the correct path.")
        raise
    except Exception as e:
        print(f"❌ Model loading error: {e}")
        raise


def load_vip_checkpoint(model: torch.nn.Module, config: Dict) -> torch.nn.Module:
    """
    Load VIP checkpoint
    
    Args:
        model (torch.nn.Module): VIP model
        config (Dict): Configuration parameters
        
    Returns:
        torch.nn.Module: Model after loading checkpoint
    """
    checkpoint_path = './checkpoints/Stage3/id0_train/vip_token.pt'
    
    print(f"📥 Loading VIP checkpoint: {checkpoint_path}")
    
    if not os.path.exists(checkpoint_path):
        print(f"⚠️  Warning: Checkpoint file does not exist: {checkpoint_path}")
        return model
    
    try:
        model.load_vip(checkpoint_path)
        print("✅ VIP checkpoint loaded successfully!")
    except Exception as e:
        print(f"❌ Checkpoint loading error: {e}")
        raise
    
    return model


def load_face_embedding_center(config: Dict) -> torch.Tensor:
    """
    Load face embedding center
    
    Args:
        config (Dict): Configuration parameters
        
    Returns:
        torch.Tensor: Face embedding center tensor
    """
    emb_center_path = './FaceDATA/FaceEmb_Center/id0.ckpt'
    
    print(f"📥 Loading face embedding center: {emb_center_path}")
    
    if not os.path.exists(emb_center_path):
        print(f"⚠️  Warning: Embedding center file does not exist: {emb_center_path}")
        return None
    
    try:
        # Specify loading to specific device
        device = torch.device(f"cuda:{config['DEVICE']}" if torch.cuda.is_available() else "cpu")
        emb_center = torch.load(emb_center_path, map_location=device)
        print("✅ Face embedding center loaded successfully!")
        return emb_center
    except Exception as e:
        print(f"❌ Embedding center loading error: {e}")
        raise


print("🔧 Model loading function definition complete!")
print("Including features:")
print("- VIP model loading function")
print("- Checkpoint loading function")
print("- Face embedding center loading function")

## 4. Load Vision-Language Models and Processors

Execute model loading to initialize core components of the VIP DFD detection system.

In [ ]:
# Load all model components
print("🚀 Starting to load VIP DFD detection system...")
print("=" * 50)

# Record start time
start_time = datetime.now()

# Load VIP models
try:
    model, processor, face_model = load_vip_models(DEFAULT_CONFIG)
    
    # Load VIP checkpoint
    model = load_vip_checkpoint(model, DEFAULT_CONFIG)
    
    # Count model parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"📊 Total trainable parameters: {total_params:,}")
    
    # Display model device information
    device_id = DEFAULT_CONFIG['DEVICE']
    print(f"📱 Model loaded on device: CUDA:{device_id}")
    
    # Verify model is on specified device
    if hasattr(model, 'device'):
        print(f"📱 Model current device: {model.device}")
    elif hasattr(model.vl_model, 'device'):
        print(f"📱 VL model current device: {model.vl_model.device}")
    
    # Record loading time
    load_time = datetime.now() - start_time
    print(f"⏱️  Model loading time: {load_time.total_seconds():.2f} seconds")
    
    print("=" * 50)
    print("✅ VIP DFD detection system loading complete!")
    
    # Display model information
    print("\n📋 Model information:")
    print(f"- Model type: VIP DFD LLM")
    print(f"- Token count: {DEFAULT_CONFIG['TOKEN_NUM']}")
    print(f"- Device: CUDA:{DEFAULT_CONFIG['DEVICE']}")
    print(f"- Face model: TransFace")
    print(f"- Trainable parameters: {total_params:,}")
    
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    print("Please check if model file paths and dependencies are correct.")
    # In actual deployment, may need to exit or provide fallback solution
    print("ℹ️  Note: In this demo environment, some model files may not be accessible.")

## 5. Load Face Recognition Model and Center Features

Load face recognition model and corresponding center feature embeddings for similarity calculation.

In [ ]:
# Load face embedding center
print("🔄 Loading face recognition related components...")
print("=" * 50)

try:
    # Get device ID
    device_id = int(DEFAULT_CONFIG['DEVICE'])
    device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
    print(f"📌 Using device: {device}")
    
    # Load face embedding center
    face_emb_center = load_face_embedding_center(DEFAULT_CONFIG)
    
    if face_emb_center is not None:
        # Ensure embedding center is on correct device
        face_emb_center = face_emb_center.to(device)
        print(f"📊 Face embedding center dimension: {face_emb_center.shape}")
        print(f"📊 Face embedding center device: {face_emb_center.device}")
        print("✅ Face recognition components loaded successfully!")
    else:
        print("⚠️  Using simulated face embedding center for demo")
        # Create simulated embedding center for demo, directly create on specified device
        face_emb_center = torch.randn(512).to(device)  # Assume embedding dimension is 512
        print(f"📊 Simulated embedding center dimension: {face_emb_center.shape}")
        print(f"📊 Simulated embedding center device: {face_emb_center.device}")
        
    print("=" * 50)
    print("✅ Face recognition system ready!")
    
    # Display face model information
    print("\n📋 Face recognition model information:")
    print(f"- Model type: TransFace")
    print(f"- Input size: {DEFAULT_CONFIG['FACE_IMG_SIZE']}")
    print(f"- Feature dimension: {DEFAULT_CONFIG['FACE_TOKEN_DIM']}")
    print(f"- Center embedding dimension: {face_emb_center.shape}")
    print(f"- Running device: {device}")
    
except Exception as e:
    print(f"❌ Face recognition component loading failed: {e}")
    print("ℹ️  Note: In this demo environment, some files may not be accessible.")

## 6. Inference Testing Function Implementation

Implement core inference testing functions for deepfake detection on images.

In [ ]:
@torch.no_grad()
def infer_single_image(img_path: str, model: torch.nn.Module, face_model: torch.nn.Module,
                      emb_center: torch.Tensor, processor, mode: str = 'cls') -> Dict:
    """
    Perform inference on a single image
    
    Args:
        img_path (str): Image path
        model (torch.nn.Module): VIP model
        face_model (torch.nn.Module): Face model
        emb_center (torch.Tensor): Center embedding
        processor: Processor
        mode (str): Inference mode
        
    Returns:
        Dict: Inference results
    """
    try:
        # Import necessary modules
        from qwen_vl_utils import process_vision_info
        
        # Get device ID
        device_id = int(DEFAULT_CONFIG['DEVICE'])
        device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
        
        # Calculate similarity score
        sim_score = calculate_similarity_with_center(img_path, emb_center, face_model)
        print(sim_score)
        # Generate messages
        messages = get_message(img_path, sim_score, mode=mode)
        
        # Prepare visual inputs
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        # If image exceeds maximum size, perform scaling (consistent with training stage)
        for i in range(len(image_inputs)):
            h, w = image_inputs[i].size
            if max(h, w) > DEFAULT_CONFIG['MAX_IMAGE_SIZE']:
                scale = DEFAULT_CONFIG['MAX_IMAGE_SIZE'] / max(h, w)
                new_w = int(w * scale)
                new_h = int(h * scale)
                image_inputs[i] = image_inputs[i].resize((new_w, new_h), Image.LANCZOS)
        
        # Prepare inference inputs
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
            our_token=True,
            our_token_length=1,
            face_pad=True,
            face_length=model.vl_model.facechecker.face_checker.vip_prompt.data.shape[0]
        )
        # Ensure inputs are on correct device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate output
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=DEFAULT_CONFIG['MAX_NEW_TOKENS'],
            do_sample=True
        )
        
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        
        # Parse results
        result_text = output_text[0]
        cls_content = result_text.split('</Conclusion>')[0]
        
        # Judge results
        prediction = None
        if 'yes' in cls_content.lower() or 'same' in cls_content.lower() or '是' in cls_content.lower():
            prediction = 'same'
        elif 'no' in cls_content.lower() or 'different' in cls_content.lower() or '否' in cls_content.lower():
            prediction = 'different'
        
        return {
            'image_path': img_path,
            'similarity_score': sim_score,
            'prediction': prediction,
            'raw_output': result_text,
            'success': True
        }
        
    except Exception as e:
        return {
            'image_path': img_path,
            'similarity_score': 0,
            'prediction': None,
            'raw_output': str(e),
            'success': False
        }

model=''
processor=''
from Models.Face_Model.FaceModel import FG_Face
face_model = FG_Face(
    attributes='',
    token_dim=DEFAULT_CONFIG['FACE_TOKEN_DIM'],
    model_name='transface'
)
face_model.cuda(int(DEFAULT_CONFIG['DEVICE']))
        
infer_single_image(img_path='./Example/with_text/id0/r_efs_i/id0_1/fake.png',
                   model=model,
                   face_model=face_model,
                   emb_center=face_emb_center,
                   processor=processor,
                   mode='cls')